# Titanic Survival Dataset 
**Domain : Disaster Survival**

# Objective of This Project :
**Demostrate professional-level data cleaning skills by taking a deliberately messy dataset and systematically transfroming it into a clean, analysis-redy dataset.**

# Feature Checklist:

* Load dataset and produce a "Data Quality Report": count of nulls per column, duplicate rows. data type issues, value range anomalies

* Missing data handling: choose an appropriate strategy for each column (mean/median imputation, mode imputation, forward fill, or row deletion) and justify each choice in a markdown cell 

* Duplicate removal: identify and duplicate rows; document how many were removed

* Standardisation: normalise inconsistent formatting

* Outlier detection: use IQR method or Z-score to identify outliers in numeric columns; decide and documnet whether to cap, remove, or retain each

* Data type correction: ensure all columns have the correct dtype 

# Import Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

# Load dataset and produce a "Data Quality Report":
**count of nulls per column, duplicate rows. data type issues, value range anomalies**

In [2]:
# Loading the dataset
df =pd.read_csv('Titanic-Dataset.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# Data Quality Report

**Shape of the dataset :**

In [3]:
df.shape

(891, 12)

**Total Records & Columns in Dataset :**
* Records : 891
* Columns : 12

# 

**Duplicate Rows in Dataset :**

In [4]:
df.duplicated().sum()

np.int64(0)

**Checking the data types of each column :**

In [5]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

**Info about Dataset :**

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB


**Value range anomalies :**

In [7]:
# Replace df with your DataFrame variable name if it's different.
checks = {
    "Invalid Survived value": df["Survived"].notna() & ~df["Survived"].isin([0, 1]),
    "Invalid passenger class": df["Pclass"].notna() & ~df["Pclass"].isin([1, 2, 3]),
    "Age outside 0–120": df["Age"].notna() & ~df["Age"].between(0, 120),
    "Negative fare": df["Fare"].notna() & df["Fare"].lt(0),
    "Negative SibSp": df["SibSp"].notna() & df["SibSp"].lt(0),
    "Negative Parch": df["Parch"].notna() & df["Parch"].lt(0),
    "Invalid Sex": df["Sex"].notna() & ~df["Sex"].isin(["male", "female"]),
    "Invalid Embarked": df["Embarked"].notna() & ~df["Embarked"].isin(["C", "Q", "S"]),
    "Fractional SibSp": df["SibSp"].notna() & (df["SibSp"] % 1 != 0),
    "Fractional Parch": df["Parch"].notna() & (df["Parch"] % 1 != 0),
}

# Collect all anomaly reasons for each row.
reasons = pd.Series("", index=df.index, dtype="object")

for reason, condition in checks.items():
    mask = condition.fillna(False)
    reasons.loc[mask] = reasons.loc[mask].apply(
        lambda existing: f"{existing}; {reason}" if existing else reason
    )

anomalies = df.loc[reasons.ne("")].copy()
anomalies["Anomaly"] = reasons.loc[reasons.ne("")]

print(anomalies)

Empty DataFrame
Columns: [PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked, Anomaly]
Index: []


**Total Null Values by Column :**

In [8]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [9]:
# Null percentage and unique counts
quality = pd.DataFrame({
    "nulls": df.isnull().sum(),
    "null_%": (df.isnull().mean() * 100).round(2),
    "unique": df.nunique(),
    "dtype": df.dtypes.astype(str)
})
print(quality)

# Value range summary
print(df[["Age", "SibSp", "Parch", "Fare"]].describe())

# Suspicious values
print("Fare == 0:", (df["Fare"] == 0).sum())

             nulls  null_%  unique    dtype
PassengerId      0    0.00     891    int64
Survived         0    0.00       2    int64
Pclass           0    0.00       3    int64
Name             0    0.00     891      str
Sex              0    0.00       2      str
Age            177   19.87      88  float64
SibSp            0    0.00       7    int64
Parch            0    0.00       7    int64
Ticket           0    0.00     681      str
Fare             0    0.00     248  float64
Cabin          687   77.10     147      str
Embarked         2    0.22       3      str
              Age       SibSp       Parch        Fare
count  714.000000  891.000000  891.000000  891.000000
mean    29.699118    0.523008    0.381594   32.204208
std     14.526497    1.102743    0.806057   49.693429
min      0.420000    0.000000    0.000000    0.000000
25%     20.125000    0.000000    0.000000    7.910400
50%     28.000000    0.000000    0.000000   14.454200
75%     38.000000    1.000000    0.000000   31.000

# Missing data handling :

In [10]:
df['Age']   = df['Age'].fillna(df['Age'].median())
df['Cabin'] = df['Cabin'].fillna("Unknown")
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

### Justification for Handling Missing Values

- **Age — Median:** Missing ages were replaced with the median because it is less affected by unusually high or low ages than the mean.
- **Cabin — "Unknown":** Missing cabin values were labeled `"Unknown"` to retain those passenger records while making the missing information explicit.
- **Embarked — Mode:** Missing embarkation ports were replaced with the most frequent value because `Embarked` is categorical.

# Duplicate Removal :

In [11]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)
print(f"Rows before: {before} | Rows after: {after} | Removed: {before - after}")

# Extra check: PassengerId must be unique
print("Duplicate PassengerIds:", df["PassengerId"].duplicated().sum())

Rows before: 891 | Rows after: 891 | Removed: 0
Duplicate PassengerIds: 0


# Standardisation :

In [12]:
# Standardisation: normalise inconsistent text formatting
text_cols = ["Name", "Sex", "Ticket", "Cabin", "Embarked"]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()          # remove extra spaces
    df[col] = df[col].str.replace(r"\s+", " ", regex=True)  # collapse double spaces

df["Sex"] = df["Sex"].str.lower()          # male / female
df["Embarked"] = df["Embarked"].str.upper()  # C / Q / S
df["Cabin"] = df["Cabin"].str.upper()
df["Name"] = df["Name"].str.title()

# Optional: readable labels for Embarked
embarked_map = {"C": "Cherbourg", "Q": "Queenstown", "S": "Southampton"}
df["Embarked_Port"] = df["Embarked"].map(embarked_map)

print(df[["Sex", "Embarked", "Embarked_Port"]].apply(lambda s: s.unique().tolist()))

Sex                                    [male, female]
Embarked                                    [S, C, Q]
Embarked_Port    [Southampton, Cherbourg, Queenstown]
dtype: object


“Text columns were trimmed and made consistently cased. Sex was lowercased and Embarked uppercased. Feature scaling (StandardScaler) is intentionally not applied, because it is a modelling step and would distort the real values used for outlier detection.”

# Outlier detection :

In [13]:
# Check continuous/count-like numeric features, not numeric-coded categories or IDs
candidate_cols = ["Age", "SibSp", "Parch", "Fare"]
outlier_cols = [col for col in candidate_cols if col in df.columns]

rows = []
outlier_masks = {}

for col in outlier_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    mask = df[col].lt(lower) | df[col].gt(upper)
    outlier_masks[col] = mask

    rows.append({
        "column": col,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": int(mask.sum()),
        "outlier_percent": round(mask.mean() * 100, 2),
    })

outlier_summary = pd.DataFrame(rows)
outlier_summary

,column,lower_bound,upper_bound,outlier_count,outlier_percent
0,Age,2.500,54.5000,66,7.41
1,SibSp,-1.500,2.5000,46,5.16
2,Parch,0.000,0.0000,213,23.91
3,Fare,-26.724,65.6344,116,13.02


In [14]:
# Decision per column
# Age   -> retain (valid ages, real passengers)
# SibSp -> retain (large families are genuine values)
# Parch -> retain (zero is the norm, so IQR over-flags)
# Fare  -> cap at the IQR upper bound (extreme skew from a few first-class fares)

q1, q3 = df["Fare"].quantile([0.25, 0.75])
upper_fare = q3 + 1.5 * (q3 - q1)

df["Fare_Capped"] = df["Fare"].clip(upper=upper_fare)

print("Fare max before:", df["Fare"].max())
print("Fare max after :", df["Fare_Capped"].max())

Fare max before: 512.3292
Fare max after : 65.6344


### Outlier Decisions
| Column | Decision | Reason |
|---|---|---|
| Age | Retain | Outliers are real elderly passengers |
| SibSp | Retain | Large families are genuine |
| Parch | Retain | Zero is the typical value, so the IQR rule over-flags |
| Fare | Cap (Winsorise) | Heavy right skew; capping reduces distortion and keeps all rows |

# Data type correction: 
**Ensure all columns have the correct dtype**

In [15]:
# Review the current types and sample values first
display(df.dtypes)
display(df.head())

# Continuous measurements; Age can be fractional and may contain missing values
for col in ["Age", "Fare"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="raise").astype("float64")

# Counts and binary target; nullable integers preserve missing values
for col in ["SibSp", "Parch", "Survived"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="raise").astype("Int64")

# Identifiers and text fields—not quantities to calculate with
for col in ["PassengerId", "Name", "Sex", "Ticket", "Cabin", "Embarked"]:
    if col in df.columns:
        df[col] = df[col].astype("string")

# Passenger class is an ordered category, not a measured quantity
if "Pclass" in df.columns:
    pclass = pd.to_numeric(df["Pclass"], errors="raise")
    if not pclass.dropna().isin([1, 2, 3]).all():
        raise ValueError("Pclass contains values outside 1, 2, and 3.")
    df["Pclass"] = pd.Categorical(
        pclass, categories=[1, 2, 3], ordered=True
    )

# Confirm the conversions
display(df.dtypes)
display(df.isna().sum())

PassengerId        int64
Survived           int64
Pclass             int64
Name                 str
Sex                  str
Age              float64
SibSp              int64
Parch              int64
Ticket               str
Fare             float64
Cabin                str
Embarked             str
Embarked_Port        str
Fare_Capped      float64
dtype: object

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Embarked_Port,Fare_Capped
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,UNKNOWN,S,Southampton,7.2500
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Cherbourg,65.6344
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,UNKNOWN,S,Southampton,7.9250
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Southampton,53.1000
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,UNKNOWN,S,Southampton,8.0500


PassengerId        string
Survived            Int64
Pclass           category
Name               string
Sex                string
Age               float64
SibSp               Int64
Parch               Int64
Ticket             string
Fare              float64
Cabin              string
Embarked           string
Embarked_Port         str
Fare_Capped       float64
dtype: object

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin            0
Embarked         0
Embarked_Port    0
Fare_Capped      0
dtype: int64

**Data type correction:** 

Age and Fare are stored as floating-point numbers; SibSp and Parch are nullable integers representing counts. Survived is a nullable integer target coded 0/1. PassengerId is stored as text because it is an identifier, not a quantity. Name, Sex, Ticket, Cabin, and Embarked are text fields. Pclass is an ordered category (1st, 2nd, 3rd class). Missing values are preserved.

In [16]:
df["Survived"] = df["Survived"].astype("int64")
df["SibSp"] = df["SibSp"].astype("int64")
df["Parch"] = df["Parch"].astype("int64")

# Final Validation :

In [17]:
# Final validation
print("Shape:", df.shape)
print("Remaining nulls:\n", df.isnull().sum())
print("Duplicates:", df.duplicated().sum())
print(df.dtypes)

# Save the cleaned dataset
df.to_csv("Titanic_Cleaned.csv", index=False)

Shape: (891, 14)
Remaining nulls:
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin            0
Embarked         0
Embarked_Port    0
Fare_Capped      0
dtype: int64
Duplicates: 0
PassengerId        string
Survived            int64
Pclass           category
Name               string
Sex                string
Age               float64
SibSp               int64
Parch               int64
Ticket             string
Fare              float64
Cabin              string
Embarked           string
Embarked_Port         str
Fare_Capped       float64
dtype: object
